In [33]:
import numpy as np
import pandas as pd
import os
print(os.getcwd())

/home/yli/MR_CDFT/MRCDFT


In [34]:
para_path = 'examples/20Ne/para.dat'
b23_path = 'examples/20Ne/b23.dat'
output_dir = 'examples/20Ne/output'
A=20


In [60]:
# extract eMax, nbeta, nphi from parameter file
with open(para_path, "r", encoding="utf-8") as f:
    for line in f:
        line_nocomment = line.split("!")[0]

        if "n0f" in line_nocomment:
            eMax = int(line_nocomment.split("=")[1].split()[0])
        if "nphi" in line_nocomment:
            nphi = int(line_nocomment.split("=")[1])
        if "nbeta" in line_nocomment:
            nbeta = int(line_nocomment.split("=")[1])
        if "AMP" in line_nocomment:
            AMP = int(line_nocomment.split("=")[1])
        if "PNP" in line_nocomment:
            PNP = int(line_nocomment.split("=")[1])
        if "Kernels" in line_nocomment:
            Kernels = int(line_nocomment.split("=")[1])
    if(AMP == 0): nbeta = 1
    if(PNP == 0): nphi = 1

eMax = 6; nphi = 1; nbeta = 1; AMP = 0; Kernels = 2

print('eMax=',eMax,' nphi=',nphi,' nbeta=',nbeta,"AMP=",AMP,"PNP=",PNP," Kernels=",Kernels)

eMax= 6  nphi= 1  nbeta= 1 AMP= 0 PNP= 1  Kernels= 2


In [62]:
oneB_ME_path = os.path.join(output_dir,f'mScheme_1B_A{A}_eMax{eMax:02d}.me')
oneB_ME = pd.read_csv(oneB_ME_path,sep=r'\s+')
oneB_ME = oneB_ME.drop(columns=["n1", "n2","l1", "l2","2j1", "2j2","2j_m1", "2j_m2"])
print(oneB_ME_path)
oneB_ME

examples/20Ne/output/mScheme_1B_A20_eMax06.me


,ifg,m1,m2,r^2,r^4,r^2Y20,r^4Y20,r^4Y40,f2,Eps1B
0,1,1,1,3.88767,25.18992,-0.00000,-0.00000,-0.0,-0.00000,2.00455
1,1,1,2,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
2,1,1,3,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
3,1,1,4,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
4,1,1,5,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...
85819,2,240,236,0.00000,0.00000,-0.00000,-0.00000,0.0,-0.00000,0.00000
85820,2,240,237,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
85821,2,240,238,0.00000,0.00000,-0.00000,-0.00000,-0.0,-3.93045,0.00000
85822,2,240,239,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000


In [63]:
# Prepare result dataframe, add TD density filename and Kernel filenames

DensList = pd.DataFrame(columns=['beta2(qf)','beta3(qf)', 'beta2(qi)','beta3(qi)', 'Ddens_filename', 'Kernel_filename'])

b23_data = pd.read_csv(b23_path,
                    sep=r'\s+',   # 按空格分隔
                    skiprows=0)
for q1 in range(len(b23_data)):
    for q2 in range(len(b23_data)):
        if(Kernels == 2 and q1 != q2): continue
        beta2q1 = b23_data.iloc[q1]['beta2']
        beta3q1 = b23_data.iloc[q1]['beta3']
        beta2q2 = b23_data.iloc[q2]['beta2']
        beta3q2 = b23_data.iloc[q2]['beta3']

        TDdens_filename = f"Proj_D1B.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.me"
        if not os.path.exists(os.path.join(output_dir,TDdens_filename)):
            print(TDdens_filename," not exist !")
        if(q1 <= q2):
            Kernel_filename = f"Proj_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        else:
            Kernel_filename = f"Proj_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}_{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}.elem"
        if not os.path.exists(os.path.join(output_dir,Kernel_filename)):
            print(Kernel_filename," not exist !")
        
        DensList.loc[len(DensList)] = [beta2q1, beta3q1, beta2q2, beta3q2, TDdens_filename, Kernel_filename]
DensList

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Ddens_filename,Kernel_filename
0,0.0,0.0,0.0,0.0,Proj_D1B.0D_eMax06.01.01+000+000_+000+000.me,Proj_kern.0D_eMax06.01.01+000+000_+000+000.elem
1,0.1,0.0,0.1,0.0,Proj_D1B.0D_eMax06.01.01+010+000_+010+000.me,Proj_kern.0D_eMax06.01.01+010+000_+010+000.elem
2,0.2,0.0,0.2,0.0,Proj_D1B.0D_eMax06.01.01+020+000_+020+000.me,Proj_kern.0D_eMax06.01.01+020+000_+020+000.elem
3,0.3,0.0,0.3,0.0,Proj_D1B.0D_eMax06.01.01+030+000_+030+000.me,Proj_kern.0D_eMax06.01.01+030+000_+030+000.elem
4,0.4,0.0,0.4,0.0,Proj_D1B.0D_eMax06.01.01+040+000_+040+000.me,Proj_kern.0D_eMax06.01.01+040+000_+040+000.elem
5,0.5,0.0,0.5,0.0,Proj_D1B.0D_eMax06.01.01+050+000_+050+000.me,Proj_kern.0D_eMax06.01.01+050+000_+050+000.elem


In [ ]:
Res_data = DensList.copy()
for q1q2 in range(len(Res_data)):
    Ddens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Ddens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read D1B density
    Ddens = pd.read_csv(Ddens_path, sep=r'\s+')

    J = 0; Pi = '+'; K1 = 0; K2 = 0
    df_tmp = Ddens[(Ddens["J"]==J) & (Ddens["Pi"]==Pi) & (Ddens["K1"]==K1) & (Ddens["K2"]==K2)]
    merged = pd.merge(df_tmp, oneB_ME, left_on=["ifg","m1", "m2"], right_on=["ifg","m2", "m1"], how="left")
    merged = merged.drop(columns=['m1_y','m2_y']).rename(columns={'m1_x':'m1','m2_x':'m2'})

    # read norm kernel
    with open(Kernel_path, 'r') as f:
        line = f.readlines()[J*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        # print('Norm=',Norm)
    # print('J=',Ji)

    Res_data.loc[q1q2,f'Norm({J}{Pi})'] = Norm
    Res_data.loc[q1q2,f'Energy({J}{Pi})'] = Energy

    # check particle number
    df_merge_N = merged[merged['m1']==merged['m2']]
    total_neutron = df_merge_N["neutron"].sum()
    total_proton = df_merge_N["proton"].sum()
    Res_data.loc[q1q2,f'N'] = total_neutron/ Norm
    Res_data.loc[q1q2,f'Z'] = total_proton/ Norm

    # calculate r^2
    total_neutron = (merged["neutron"] * merged["r^2"]).sum()
    total_proton = (merged["proton"] * merged["r^2"]).sum()
    Res_data.loc[q1q2,f'r2(n)'] = total_neutron/Norm
    Res_data.loc[q1q2,f'r2(p)'] = total_proton/Norm
    Res_data.loc[q1q2,f'r2(total)'] = total_neutron/Norm + total_proton/Norm
    

Res_data.drop(columns=['Ddens_filename', 'Kernel_filename'])


,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Norm(0+),Energy(0+),N,Z,r2(n),r2(p),r2(total),Eps1B(n),Eps1B(p),Eps1B(total)
0,0.0,0.0,0.0,0.0,1.0,-150.95830,10.0,10.0,76.960734,78.335021,155.295755,68.718401,71.338622,140.057023
1,0.1,0.0,0.1,0.0,1.0,-151.22265,10.0,10.0,77.027752,78.397575,155.425328,62.404524,64.856506,127.261030
2,0.2,0.0,0.2,0.0,1.0,-151.96494,10.0,10.0,77.256853,78.615678,155.872531,56.586266,58.835392,115.421658
3,0.3,0.0,0.3,0.0,1.0,-152.87231,10.0,10.0,77.879115,79.232586,157.111701,51.717748,53.726769,105.444517
4,0.4,0.0,0.4,0.0,1.0,-153.62448,10.0,10.0,78.998031,80.353342,159.351372,47.849635,49.665982,97.515617
5,0.5,0.0,0.5,0.0,1.0,-154.13674,10.0,10.0,80.582682,81.948560,162.531242,44.762401,46.384277,91.146677


In [65]:
import math
Res_data = DensList.copy()
for q1q2 in range(len(Res_data)):
    Ddens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Ddens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read D1B density
    Ddens = pd.read_csv(Ddens_path, sep=r'\s+')

    J = 0; Pi = '+'; K1 = 0; K2 = 0
    df_tmp = Ddens[(Ddens["J"]==J) & (Ddens["Pi"]==Pi) & (Ddens["K1"]==K1) & (Ddens["K2"]==K2)]
    merged = pd.merge(df_tmp, oneB_ME, left_on=["ifg","m1", "m2"], right_on=["ifg","m2", "m1"], how="left")
    merged = merged.drop(columns=['m1_y','m2_y']).rename(columns={'m1_x':'m1','m2_x':'m2'})

    # read norm kernel
    with open(Kernel_path, 'r') as f:
        line = f.readlines()[J*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        # print('Norm=',Norm)
    # print('J=',Ji)

    # calculate r^2
    total_neutron = (merged["neutron"] * merged["r^2"]).sum()
    total_proton = (merged["proton"] * merged["r^2"]).sum()
    Res_data.loc[q1q2,f'r2(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4
    total_neutron = (merged["neutron"] * merged["r^4"]).sum()
    total_proton = (merged["proton"] * merged["r^4"]).sum()
    Res_data.loc[q1q2,f'r4(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^2 Y20
    total_neutron = (merged["neutron"] * merged["r^2Y20"]).sum()
    total_proton = (merged["proton"] * merged["r^2Y20"]).sum()
    Res_data.loc[q1q2,f'r2Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4 Y20
    total_neutron = (merged["neutron"] * merged["r^4Y20"]).sum()
    total_proton = (merged["proton"] * merged["r^4Y20"]).sum()
    Res_data.loc[q1q2,f'r4Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4 Y40
    total_neutron = (merged["neutron"] * merged["r^4Y40"]).sum()
    total_proton = (merged["proton"] * merged["r^4Y40"]).sum()
    Res_data.loc[q1q2,f'r4Y40(total)'] = total_neutron/Norm + total_proton/Norm
    
    # calculate r^2_perp = 2/3 r^2 - 4/3*sqrt(pi/5)r^2 Y20
    Res_data['r^2_perp'] = 2.0/3.0 * Res_data['r2(total)'] - 4.0/3.0*math.sqrt(math.pi/5.0)*Res_data['r2Y20(total)'] 
    # calculate r^4_perp = 8/15 r^4 - 32/21*sqrt(pi/5)r^4 Y20 + 16/105*sqrt(pi)*r^4 Y40
    Res_data['r^4_perp'] = 8.0/15.0 * Res_data['r4(total)'] - 32.0/21.0*math.sqrt(math.pi/5.0)*Res_data['r4Y20(total)'] + 16.0/105.0*math.sqrt(math.pi)*Res_data['r4Y40(total)']
     
Res_data.drop(columns=['Ddens_filename', 'Kernel_filename'])


,beta2(qf),beta3(qf),beta2(qi),beta3(qi),r2(total),r4(total),r2Y20(total),r4Y20(total),r4Y40(total),r^2_perp,r^4_perp
0,0.0,0.0,0.0,0.0,155.295755,1760.708909,3.407373e-07,-0.000053,-0.000646,103.530503,939.044642
1,0.1,0.0,0.1,0.0,155.425328,1764.841941,5.065903e+00,74.084595,5.515999,98.262796,853.254194
2,0.2,0.0,0.2,0.0,155.872531,1779.186477,1.013181e+01,150.454564,24.810772,93.206838,773.870821
3,0.3,0.0,0.3,0.0,157.111701,1817.393279,1.519772e+01,232.220841,67.412550,88.678860,706.990890
4,0.4,0.0,0.4,0.0,159.351372,1885.886917,2.026362e+01,321.144436,133.303973,84.817881,653.909078
5,0.5,0.0,0.5,0.0,162.531242,1983.288360,2.532952e+01,417.251309,213.351884,81.583710,611.391790
